# AI Newsroom Studio — Pipeline Notebook
10-agent pipeline: HackerNews trends → fact-check → script → video → YouTube Shorts.

**Agents built:** Agent 1 (Trend Hunter) · Agent 2 (Context Researcher) · Agent 3 (Fact Checker)

---
## Setup: Shared State

In [1]:
from typing_extensions import TypedDict

class NewsroomState(TypedDict):
    stories: dict   # keyed by story_id (slug), enriched in place by each agent

---
## Agent 1: Trend Fetcher
Fetches top HackerNews stories and scores them by velocity.

**Velocity** = (upvotes + comments×2) / age_hrs — rewards recent engagement.

In [2]:
from agents.agent1 import fetch_trends
import re

def make_story_id(title: str) -> str:
    """Slugify title to a stable dict key. e.g. 'Melody India Italy' → 'melody-india-italy'"""
    return re.sub(r'[^a-z0-9]+', '-', title.lower()).strip('-')

# AGENT 1 NODE
def trend_hunter_node(state: NewsroomState) -> dict:
    trends = fetch_trends()
    stories = {make_story_id(t["title"]): t for t in trends}
    return {"stories": stories}

In [3]:
# Run Agent 1
call = trend_hunter_node(NewsroomState(stories={}))
print(f"Stories fetched: {len(call['stories'])}")
for sid, s in call['stories'].items():
    print(f"  {s['velocity']:>6.1f} vel | {sid}")

Stories fetched: 8
    91.5 vel | fable-turned-remarkable-into-tom-riddle-s-diary-from-harry-potter
    69.5 vel | openwrt-one-open-hardware-router
    68.3 vel | glm-5-2-and-the-coming-ai-margin-collapse
    65.0 vel | microsoft-can-track-users-via-a-windows-device-id
    53.2 vel | comaps-foss-offline-maps
    40.3 vel | how-to-sequence-your-own-dna-at-home
    24.2 vel | small-ai-models-gain-traction-in-places-with-unreliable-networks
     1.3 vel | dolosse-a-south-african-invention-used-over-the-world


In [4]:
# ── INSPECT after Agent 1 ────────────────────────────────────────────
# Agent 1 fields: title, url, source, category, engagement, velocity
from urllib.parse import urlparse

print(f"{'VEL':>7} {'ENGAGEMENT':>11} {'CATEGORY':<14} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call['stories'].items():
    vel        = story.get('velocity', 0)
    engagement = story.get('engagement', 0)
    category   = story.get('category', '?')
    domain     = urlparse(story.get('url','')).netloc.replace('www.','')
    title      = story.get('title','')[:45]
    print(f"  {vel:>6.1f}  {engagement:>10}  {category:<14} {domain:<28} {title}")

    VEL  ENGAGEMENT CATEGORY       SOURCE_DOMAIN                TITLE
----------------------------------------------------------------------------------------------------
    91.5         783  technology     github.com                   Fable turned reMarkable into Tom Riddle's dia
    69.5         897  technology     openwrt.org                  OpenWrt One – Open Hardware Router
    68.3         725  technology     martinalderson.com           GLM 5.2 and the coming AI margin collapse
    65.0         105  technology     pcmag.com                    Microsoft Can Track Users via a Windows Devic
    53.2         719  technology     comaps.app                   CoMaps – FOSS Offline Maps
    40.3         333  technology     bradleywoolf.com             How to sequence your own DNA at home
    24.2         210  technology     spectrum.ieee.org            Small AI Models Gain Traction In places with 
     1.3          74  technology     thisbugslife.com             Dolosse – a South Afri

---
## Agent 2: Context Researcher
For each story:
1. Fetches article content (3-tier: trafilatura → Jina → Tavily)
2. Gathers background (DDG news + Wikipedia)
3. Synthesises one tight background paragraph (qwen2.5:3b, content-anchored)

**Key design:**  rejects page chrome before accepting fetched text.
Backgrounds are honest-empty when DDG has no coverage (GitHub/arXiv/new tools — see KNOWN_ISSUES.md).

In [5]:
import ollama, subprocess, time

# Health-check: Ollama must be running for Agent 2 synthesis
try:
    ollama.list()
    print("✅ Ollama already running")
except Exception:
    try:
        subprocess.Popen(["ollama", "serve"])
        time.sleep(3)
        ollama.list()
        print("✅ Ollama started")
    except Exception:
        print("✗ Run  in a terminal manually")

✅ Ollama already running


In [6]:
from agents.agent2 import fetch_url_content, fetch_trend_background

# AGENT 2 NODE
def context_researcher_node(state: NewsroomState) -> dict:
    """Enriches each story in-place with content + background keys."""
    stories = state["stories"]
    for sid, story in stories.items():
        print(f"Agent 2 Starts For : {sid}")
        story["content"]    = fetch_url_content(story["url"])
        story["background"] = fetch_trend_background(story["title"], story["content"])
        print(f"  Agent 2 Ends for: {story['title']}")
        print("=" * 80)
    return {"stories": stories}

In [7]:
# Run Agent 2 (enriches call['stories'] in-place)
call2 = context_researcher_node(call)

Agent 2 Starts For : fable-turned-remarkable-into-tom-riddle-s-diary-from-harry-potter
Trafiltura Success ✅  In Loading Content :6835 characters
  [background] gathering snippets for: Fable turned reMarkable into Tom Riddle's diary from Harry Potter
  [bg] DDG returned 3 snippets
  [bg] Wikipedia search returns: 1459 chars
  [background] 4 snippets, synthesizing (content-anchored)...
  [synth] 4 snippets, 2319 snippet chars → llama3.1:8b
  [synth] llama3.1:8b → 484 chars
Result After Background Syntezing is : A Canadian developer has created a mod for the reMarkable Paper Pro tablet that transforms it into an interactive version of Tom Riddle's diary from the Harry Potter series. The mod uses Anthropic's Fable 5 AI to read handwriting and respond with sentences, creating a magical experience where users can write and receive replies in real-time. This project is not affiliated with reMarkable AS and has only been tested on the reMarkable Paper Pro tablet running OS versions 3.26-3.27.


In [8]:
# Run this in notebook after Agent 2 completes
first = next(iter(call2['stories'].values()))
print("Keys after Agent 2:", list(first.keys()))

Keys after Agent 2: ['title', 'url', 'source', 'category', 'engagement', 'velocity', 'content', 'background']


In [9]:
# ── INSPECT after Agent 2 ────────────────────────────────────────────
# Agent 1 fields + Agent 2 new fields: content, background
from urllib.parse import urlparse

# Agent 1 block (unchanged)
print('── Agent 1 fields ──')
print(f"{'VEL':>7} {'ENGAGEMENT':>11} {'CATEGORY':<14} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call2['stories'].items():
    vel        = story.get('velocity', 0)
    engagement = story.get('engagement', 0)
    category   = story.get('category', '?')
    domain     = urlparse(story.get('url','')).netloc.replace('www.','')
    title      = story.get('title','')[:45]
    print(f"  {vel:>6.1f}  {engagement:>10}  {category:<14} {domain:<28} {title}")

# Agent 2 block (new fields only)
print()
print('── Agent 2 new fields ──')
print(f"{'VEL':>7} {'CONTENT':>9} {'BG':>6}  {'BG_STATUS':<10} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call2['stories'].items():
    vel    = story.get('velocity', 0)
    clen   = len(story.get('content', ''))
    bglen  = len(story.get('background', ''))
    domain = urlparse(story.get('url','')).netloc.replace('www.','')
    title  = story.get('title','')[:45]
    bg_st  = '✅ rich' if bglen > 400 else ('⚠️ thin' if bglen > 0 else '❌ empty')
    print(f"  {vel:>6.1f}  {clen:>7}c  {bglen:>4}c  {bg_st:<10} {domain:<28} {title}")

── Agent 1 fields ──
    VEL  ENGAGEMENT CATEGORY       SOURCE_DOMAIN                TITLE
----------------------------------------------------------------------------------------------------
    91.5         783  technology     github.com                   Fable turned reMarkable into Tom Riddle's dia
    69.5         897  technology     openwrt.org                  OpenWrt One – Open Hardware Router
    68.3         725  technology     martinalderson.com           GLM 5.2 and the coming AI margin collapse
    65.0         105  technology     pcmag.com                    Microsoft Can Track Users via a Windows Devic
    53.2         719  technology     comaps.app                   CoMaps – FOSS Offline Maps
    40.3         333  technology     bradleywoolf.com             How to sequence your own DNA at home
    24.2         210  technology     spectrum.ieee.org            Small AI Models Gain Traction In places with 
     1.3          74  technology     thisbugslife.com             D

---
## Agent 3: Fact Checker
Scores each story's credibility on a **-1 to +1 scale**.
Zero is the natural boundary — negative = discard, positive = keep.

| Signal | Base Weight | How |
|--------|-------------|-----|
| `source_score` | 20% | Domain trust map (32 domains, 0.0 to +0.95) |
| `llm_credibility_check` | 60% | gpt-oss-120b → REAL +0.9 / OPINION +0.1 / SPAM -0.7 |
| `cross_verify` | 20% | Exa → DDG → confirms or contradicts via second source |

**Dynamic reweighting** — weights shift when cross-verify fires:
- Contradiction detected → llm 60%→30%, verify 20%→50%
- Confirmation detected  → llm 60%→50%, verify 20%→30%
- Neutral (not found)    → standard weights unchanged

**Key design decisions:**
- `-1 to +1` range — negative range earned by SPAM or credible contradiction
- Threshold = 0.0 — zero is the natural semantic boundary
- SPAM → -0.7 (only genuine negative LLM signal)
- Empty response / Groq failure → 0.0 neutral (never discard on crash)
- Thin content < 500 chars → 0.0 neutral
- Soft discard — story marked `discarded=True`, never deleted (audit trail)
- Cross-verify: Exa (semantic, HN-aware) → DDG fallback → neutral
- Contradiction check uses `compound-mini` (separate quota from 120b)

In [10]:
import time                                    # ← add this line
from agents.agent3 import (
    source_score,
    llm_credibility_check,
    check_credibility,
    cross_verify,
)
print("✅ Agent 3 imported")

✅ Agent 3 imported


In [11]:
# AGENT 3 NODE
def fact_checker_node(state: NewsroomState) -> dict:
    stories = state["stories"]
    for sid, story in stories.items():
        check_credibility(story)
        flag    = "🗑️ DISCARD" if story["discarded"] else "✅ KEEP"
        regime  = story.get("_cred_regime", "?")
        vby     = story.get("verified_by", "NONE")
        contra  = "❌ contradicted" if story.get("contradicted") else ""
        print(f"{story['credibility_score']:+.2f} {flag} "
              f"[{regime}] verified={vby} {contra}| {story['title']}")
        time.sleep(1)
    return {"stories": stories}

In [12]:
# Run Agent 3 on Agent 2's output (no re-run of A1 or A2)
print("── CREDIBILITY RESULTS ─────────────────────────────")
call3 = fact_checker_node(call2)
print("────────────────────────────────────────────────────")

── CREDIBILITY RESULTS ─────────────────────────────
  [llm_cred_check DEBUG] raw='REAL'
  [llm_cred_check] "Fable turned reMarkable into Tom Riddle'" → 'REAL' → 0.9
  [verify] neutral — no credible source found or both tiers failed
  [cred] regime=neutral | src=0.95×0.2 + llm=0.90×0.6 + verify=0.00×0.2 = 0.73
+0.73 ✅ KEEP [neutral] verified=NONE | Fable turned reMarkable into Tom Riddle's diary from Harry Potter
  [llm_cred_check DEBUG] raw='REAL'
  [llm_cred_check] 'OpenWrt One – Open Hardware Router' → 'REAL' → 0.9
  [verify] neutral — no credible source found or both tiers failed
  [cred] regime=neutral | src=0.00×0.2 + llm=0.90×0.6 + verify=0.00×0.2 = 0.54
+0.54 ✅ KEEP [neutral] verified=NONE | OpenWrt One – Open Hardware Router
  [llm_cred_check DEBUG] raw='OPINION'
  [llm_cred_check] 'GLM 5.2 and the coming AI margin collaps' → 'OPINION' → 0.1
  [verify] neutral — no credible source found or both tiers failed
  [cred] regime=neutral | src=0.00×0.2 + llm=0.10×0.6 + verify=0.00×0.

In [13]:
# Run this after Agent 3 completes
first = next(iter(call3['stories'].values()))
print("Keys after Agent 3:", list(first.keys()))

Keys after Agent 3: ['title', 'url', 'source', 'category', 'engagement', 'velocity', 'content', 'background', 'credibility_score', 'verified_by', 'contradicted', 'discarded', '_cred_regime', '_weights_used']


In [14]:
for sid, story in call3["stories"].items():
    print(sid)
    print(story)

fable-turned-remarkable-into-tom-riddle-s-diary-from-harry-potter
{'title': "Fable turned reMarkable into Tom Riddle's diary from Harry Potter", 'url': 'https://github.com/MaximeRivest/Riddle', 'source': 'hackernews', 'category': 'technology', 'engagement': 783, 'velocity': 91.5, 'content': 'Write on the page with your pen. After a pause, the diary drinks your ink — your words fade into the paper — the page thinks for a moment, and an answer writes itself back in a flowing hand, stroke by stroke, then fades away.\nNo screen glow, no keyboard, no chat UI. Just ink appearing on paper.\nThis is the diary from the demo.\nYou need a reMarkable Paper Pro in developer mode with a launcher installed. If that sounds like a lot, it isn\'t — remagic walks you through turning on developer mode and sets up everything with one command. Come back here, drop riddle in, and start writing to Tom.\nAlready have xovi + AppLoad? Install from the remagic catalog, grab the prebuilt bundle, or build from sour

In [15]:
# ── INSPECT after Agent 3 — cumulative (A1 + A2 + A3 fields) ────────
# Keys added: credibility_score, verified_by, contradicted,
#             discarded, _cred_regime, _weights_used
from urllib.parse import urlparse

print('── Agent 1 fields ──')
print(f"{'VEL':>7} {'ENGAGEMENT':>11} {'CATEGORY':<14} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call3['stories'].items():
    vel        = story.get('velocity', 0)
    engagement = story.get('engagement', 0)
    category   = story.get('category', '?')
    domain     = urlparse(story.get('url','')).netloc.replace('www.','')
    title      = story.get('title','')[:45]
    print(f"  {vel:>6.1f}  {engagement:>10}  {category:<14} {domain:<28} {title}")

print()
print('── Agent 2 new fields ──')
print(f"{'VEL':>7} {'CONTENT':>9} {'BG':>6}  {'BG_STATUS':<10} {'SOURCE_DOMAIN':<28} TITLE")
print('-' * 100)
for sid, story in call3['stories'].items():
    vel    = story.get('velocity', 0)
    clen   = len(story.get('content', ''))
    bglen  = len(story.get('background', ''))
    domain = urlparse(story.get('url','')).netloc.replace('www.','')
    title  = story.get('title','')[:45]
    bg_st  = '✅ rich' if bglen > 400 else ('⚠️ thin' if bglen > 0 else '❌ empty')
    print(f"  {vel:>6.1f}  {clen:>7}c  {bglen:>4}c  {bg_st:<10} {domain:<28} {title}")

print()
print('── Agent 3 new fields ──')
print(f"{'FLAG':<4} {'SCORE':<7} {'VEL':>6} {'REGIME':<14} {'WEIGHTS':<12} "
      f"{'VERIFIED_BY':<22} {'SOURCE_DOMAIN':<25} TITLE")
print('-' * 120)
for sid, story in call3['stories'].items():
    flag    = '🗑️' if story.get('discarded') else '✅'
    score   = story.get('credibility_score', 0)
    vel     = story.get('velocity', 0)
    regime  = story.get('_cred_regime', '?')
    weights = story.get('_weights_used', '?')
    vby     = story.get('verified_by', 'NONE')
    contra  = '❌ ' if story.get('contradicted') else ''
    domain  = urlparse(story.get('url','')).netloc.replace('www.','')
    title   = story.get('title','')[:40]
    print(f"{flag}  {score:+.2f}  {vel:>6.1f}  {regime:<14} {weights:<12} "
          f"{contra}{vby:<22} {domain:<25} {title}")


── Agent 1 fields ──
    VEL  ENGAGEMENT CATEGORY       SOURCE_DOMAIN                TITLE
----------------------------------------------------------------------------------------------------
    91.5         783  technology     github.com                   Fable turned reMarkable into Tom Riddle's dia
    69.5         897  technology     openwrt.org                  OpenWrt One – Open Hardware Router
    68.3         725  technology     martinalderson.com           GLM 5.2 and the coming AI margin collapse
    65.0         105  technology     pcmag.com                    Microsoft Can Track Users via a Windows Devic
    53.2         719  technology     comaps.app                   CoMaps – FOSS Offline Maps
    40.3         333  technology     bradleywoolf.com             How to sequence your own DNA at home
    24.2         210  technology     spectrum.ieee.org            Small AI Models Gain Traction In places with 
     1.3          74  technology     thisbugslife.com             D

In [16]:
from urllib.parse import urlparse
# ── INSPECT: full story details after all 3 agents ──────────────────
print(f"{'FLAG':<4} {'SCORE':<7} {'VEL':>6} {'CONTENT':>8} {'BG':>5} {'REGIME':<14} {'VERIFIED_BY':<19} {'ORIGINAL SOURCE':<25} {'TITLE':<50}")
print("-" * 145)



for sid, story in call3["stories"].items():
    flag    = "🗑️" if story.get("discarded") else "✅"
    score   = story.get("credibility_score", 0)
    vel     = story.get("velocity", 0)
    clen    = len(story.get("content", ""))
    bglen   = len(story.get("background", ""))
    regime  = story.get("_cred_regime", "?")
    vby     = story.get("verified_by", "NONE")
    contra  = "❌" if story.get("contradicted") else ""
    #weights = story.get("_weights_used", "?")
    title   = story.get("title", "")
    source = story.get("url", "")
    source_domain = urlparse(
            source
        ).netloc.replace("www.", "")


    print(    f"{flag}  {score:+.2f}  {vel:>6.1f}  {clen:>6}c  "
            f"{bglen:>4}c  {regime:<14} {contra}{vby:<20}"
            f"{source_domain:<25} {title:<50}")

FLAG SCORE      VEL  CONTENT    BG REGIME         VERIFIED_BY         ORIGINAL SOURCE           TITLE                                             
-------------------------------------------------------------------------------------------------------------------------------------------------
✅  +0.73    91.5    6000c   484c  neutral        NONE                github.com                Fable turned reMarkable into Tom Riddle's diary from Harry Potter
✅  +0.54    69.5    6000c   458c  neutral        NONE                openwrt.org               OpenWrt One – Open Hardware Router                
✅  +0.06    68.3    6000c   432c  neutral        NONE                martinalderson.com        GLM 5.2 and the coming AI margin collapse         
✅  +0.54    65.0    5942c   511c  neutral        NONE                pcmag.com                 Microsoft Can Track Users via a Windows Device ID 
✅  +0.69    53.2     954c   737c  confirmation   github.com          comaps.app                CoMaps – FOSS

### Save story : till agent 3 (quality data progress saved to disk) so even if some changes comes in agent 1 to agent 3 , agent 4+ code developement will not be affect

In [13]:

# Save to disk — skip re-processing next time
#from agent_tools.story_cache import save_stories
#save_stories(call3["stories"])

# TO track log of particular function after a hit of particular no. of times pop-up will come
